In [1]:
"""Sage-Decoder notebook import bootstrap."""
from pathlib import Path
import sys

def _find_repo_root(start):
    for candidate in (start, *start.parents):
        if (candidate / "isomorphism").is_dir() and (candidate / "pyproject.toml").is_file():
            return candidate
    raise RuntimeError("Could not find the Sage-Decoder repository root.")

repo_root = _find_repo_root(Path.cwd())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))


# 6.6.6 color code

This notebook reproduces the 6.6.6 color-code example from `Decoupling_topological_CSS_codes.pdf`. The input is the two-polynomial excitation map determined by `f = 1 + x + xy` and `g = 1 + y + xy`.

The workflow checks four objects separately: the CSS excitation map, the finite translation representation, the period table, and the decoupling result. Computation cells are kept separate from display cells so diagnostic output cannot mix with the values that should be checked.

In [2]:
from sage.all import Matrix, identity_matrix

from isomorphism import (
    R,
    build_quotient_translation_action,
    choose_smallest_oblique_cell,
    construct_excitation_map,
    decouple_coarse_matrix,
    oblique_coarse_grain,
    periods_from_generators,
    periods_from_translation_action,
    x,
    y,
)
from isomorphism.css import check_commutation

The first block constructs the CSS excitation map from the pair `(f, g)`. The commutation assertion verifies that the resulting matrix is a valid CSS excitation map before any decoupling algorithm is applied.

In [3]:
f_666 = 1 + x + x * y
g_666 = 1 + y + x * y

epsilon_666 = construct_excitation_map(f_666, g_666)
assert check_commutation(epsilon_666, num_qubits=2)

In [4]:
epsilon_666

[         x*y + x + 1          x*y + y + 1                    0                    0]
[                   0                    0 1 + y^-1 + x^-1*y^-1 1 + x^-1 + x^-1*y^-1]

The next computation constructs the finite translation representation for the one-row check matrix `[f, g]`. The display cell checks that the actions of `x` and `y` agree and have order three, matching the expected 6.6.6 color-code period.

In [5]:
check_matrix_666 = Matrix(R, [[f_666, g_666]])
representation_666 = build_quotient_translation_action(check_matrix_666, diagnostics=True)

In [6]:
identity_666 = identity_matrix(representation_666.tx.base_ring(), representation_666.tx.nrows())
assert representation_666.tx == representation_666.ty
assert representation_666.tx**3 == identity_666

(
    representation_666.basis,
    representation_666.tx,
    representation_666.ty,
    representation_666.diagnostics,
)

(
            [0 1]  [0 1]                                                                                                     
((d), (1)), [1 1], [1 1], {'generator_count': 4, 'standard_basis_size': 4, 'quotient_dimension': 2, 'monomial_basis_size': 2}
)

This block computes the period table directly from the ideal `(f, g)`. The expected square period is `L = 3`; the displayed array marks all nonzero period vectors in the square search window.

In [7]:
periods_666 = periods_from_generators(f_666, g_666, max_period=3)
assert periods_666.square_period == 3
assert (3, 0) in periods_666.vectors
assert (2, 1) in periods_666.vectors

periods_666.vectors, periods_666.square_period

(((0, 3), (1, 2), (2, 1), (3, 0), (3, 3)), 3)

The final block runs the decoupling algorithm with oblique coarse graining. The result display reports the evaluated ranks, the selected period cell, and the shape of the directly constructed inverse degree-one map.

In [8]:
cell_666 = choose_smallest_oblique_cell(periods_666.vectors)
coarse_epsilon_666 = oblique_coarse_grain(epsilon_666, *cell_666)
result_666 = decouple_coarse_matrix(
    coarse_epsilon_666,
    num_x_checks=coarse_epsilon_666.nrows() // 2,
    num_qubits=coarse_epsilon_666.ncols() // 2,
)

In [9]:
(
    (result_666.diagnostics["product_x_rank"], result_666.diagnostics["product_z_rank"]),
    cell_666,
    result_666.inverse_maps.phi1_inverse.nrows(),
    result_666.inverse_maps.phi1_inverse.ncols(),
)

((1, 1), ((1, 2), (2, 1)), 6, 6)

In [10]:
(result_666.inverse_maps.phi0_inverse.inverse()) * (coarse_epsilon_666[:3,:6]) * (result_666.inverse_maps.phi1_inverse)

[    1     0     0     0     0     0]
[    0     0 x + 1 y + 1     0     0]
[    0     0     0     0 x + 1 y + 1]

In [11]:
# Supplement QCA check: diag(phi1_inverse, phi1_dagger).
from isomorphism.chain_maps.decoupling import (
    qca_decoupled_excitation_matrix,
    qca_symplectic_matrix,
    stabilizer_redefined_excitation_matrix,
    target_excitation_matrix,
    verify_qca_decoupling,
)

qca_matrix_666 = qca_symplectic_matrix(result_666.maps, result_666.inverse_maps)
qca_product_666 = qca_decoupled_excitation_matrix(
    coarse_epsilon_666, result_666.maps, result_666.inverse_maps
)
target_epsilon_666 = target_excitation_matrix(result_666)
redefined_product_666 = stabilizer_redefined_excitation_matrix(result_666)

assert redefined_product_666 == target_epsilon_666
assert verify_qca_decoupling(result_666)

{
    "qca_matrix_shape": (qca_matrix_666.nrows(), qca_matrix_666.ncols()),
    "raw_qca_product_matches_target": qca_product_666 == target_epsilon_666,
    "after_stabilizer_redefinition_matches_target": redefined_product_666 == target_epsilon_666,
}


{'qca_matrix_shape': (12, 12),
 'raw_qca_product_matches_target': False,
 'after_stabilizer_redefinition_matches_target': True}